# Étape 4 — Inférence & Comparaison Teacher / Student
**Projet** : Single-Source Fast Generation (GENAI ING3)

On compare côte à côte :
- **Teacher** : SinFusion DDPM, **50 étapes DDIM** (lent, haute qualité de référence)
- **Student** : Générateur GAN distillé, **1 seule étape** (rapide)

Les deux modèles reçoivent **exactement les mêmes bruits z_T** → comparaison directe et équitable.

```
z_T ──── Teacher DDIM (50 steps) ────► x_teacher
z_T ──── Student GAN  ( 1 step ) ────► x_student
```

**Objectif** : mesurer le gain de vitesse et évaluer le compromis qualité/vitesse.

## 1. Setup

In [ ]:
import os, sys, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

try:
    import lpips
except ImportError:
    os.system('pip install lpips -q')
    import lpips

PROJECT_ROOT  = os.path.dirname(os.path.abspath(''))
SINFUSION_DIR = os.path.join(PROJECT_ROOT, 'sinfusion')
os.chdir(SINFUSION_DIR)
if SINFUSION_DIR not in sys.path:
    sys.path.insert(0, SINFUSION_DIR)

from models.nextnet import NextNet
from diffusion.diffusion import Diffusion

# ── À CHANGER pour une nouvelle image ────────────────────────────────────────
IMAGE_NAME = 'fruit.png'
# ─────────────────────────────────────────────────────────────────────────────
RUN_NAME = os.path.splitext(IMAGE_NAME)[0] + '_teacher'

TEACHER_CKPT = os.path.join(PROJECT_ROOT, 'teacher_checkpoint', IMAGE_NAME, 'last.ckpt')
STUDENT_CKPT = os.path.join(PROJECT_ROOT, 'outputs', IMAGE_NAME, RUN_NAME,
                             'distillation', 'checkpoints', 'student_final.pt')
OUT_DIR      = os.path.join(PROJECT_ROOT, 'outputs', IMAGE_NAME, RUN_NAME, 'inference')
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

assert os.path.exists(TEACHER_CKPT), f'Teacher checkpoint introuvable : {TEACHER_CKPT}'
assert os.path.exists(STUDENT_CKPT), f'Student checkpoint introuvable : {STUDENT_CKPT}'
print('\nCheckpoints trouvés ✓')

## 2. Charger le Teacher (SinFusion DDPM)

In [ ]:
NETWORK_DEPTH       = 16
NETWORK_FILTERS     = 64
DIFFUSION_TIMESTEPS = 50
NOISE_SCHEDULE      = 'cosine'
TRAINING_TARGET     = 'x0'

ckpt_teacher = torch.load(TEACHER_CKPT, map_location=device)
backbone     = NextNet(in_channels=3, filters_per_layer=NETWORK_FILTERS, depth=NETWORK_DEPTH)
teacher      = Diffusion(
    model=backbone,
    training_target=TRAINING_TARGET,
    timesteps=DIFFUSION_TIMESTEPS,
    noise_schedule=NOISE_SCHEDULE,
)
teacher.load_state_dict(ckpt_teacher['state_dict'])
teacher = teacher.eval().to(device)

# Dimensions de l'image source
img_source = Image.open(os.path.join(SINFUSION_DIR, 'images', IMAGE_NAME))
H, W = img_source.size[1], img_source.size[0]
print(f'Teacher chargé — image source : {W}×{H} px')
print(f'Paramètres teacher : {sum(p.numel() for p in teacher.parameters())/1e6:.2f} M')

## 3. Charger le Student (GAN distillé)

In [ ]:
class Generator(nn.Module):
    """Générateur one-step : z_T → x_0 en une seule passe forward."""
    def __init__(self, filters=64, depth=16):
        super().__init__()
        self.net = NextNet(in_channels=3, filters_per_layer=filters, depth=depth)

    def forward(self, z_T):
        t = torch.zeros(z_T.shape[0], dtype=torch.long, device=z_T.device)
        return self.net(z_T, t).clamp(-1., 1.)


ckpt_student = torch.load(STUDENT_CKPT, map_location=device)
G = Generator(filters=64, depth=16).to(device)
G.load_state_dict(ckpt_student['G_state'])
G = G.eval()

print(f'Student chargé — epoch : {ckpt_student["epoch"]}  |  step : {ckpt_student["step"]}')
print(f'Paramètres student : {sum(p.numel() for p in G.parameters())/1e6:.2f} M')

## 4. DDIM pour le Teacher

On réutilise l'implémentation correcte pour un modèle `x0` (identique au notebook 02).

In [ ]:
@torch.no_grad()
def ddim_sample_x0(diffusion, z_T, step_size=1):
    """DDIM déterministe pour modèle x0. step_size=1 → 50 étapes complètes."""
    seq      = list(range(0, diffusion.num_timesteps, step_size))
    seq_next = [-1] + seq[:-1]
    x_t = z_T
    for t, t_next in zip(reversed(seq), reversed(seq_next)):
        t_tensor      = torch.full((x_t.shape[0],), t, dtype=torch.int64, device=x_t.device)
        predicted_x0  = diffusion.model(x_t, t_tensor).clamp(-1., 1.)
        if t_next < 0:
            x_t = predicted_x0
        else:
            alpha_t      = diffusion.alphas_hat[t]
            alpha_t_next = diffusion.alphas_hat[t_next]
            direction    = (torch.sqrt(1. - alpha_t_next)
                            * (x_t - torch.sqrt(alpha_t) * predicted_x0)
                            / torch.sqrt(1. - alpha_t))
            x_t = torch.sqrt(alpha_t_next) * predicted_x0 + direction
    return x_t

print('Fonction DDIM définie ✓')

## 5. Benchmark de vitesse

On mesure le temps de génération pour **N_BENCH images**, en batch de 1 (cas réaliste d'utilisation).

Le gain de vitesse (speedup) est le rapport `t_teacher / t_student`.

In [ ]:
N_BENCH = 10  # nombre d'images pour le benchmark

# Préchauffage GPU (évite de mesurer les initialisations CUDA)
_ = ddim_sample_x0(teacher, torch.randn(1, 3, H, W, device=device))
with torch.no_grad():
    _ = G(torch.randn(1, 3, H, W, device=device))
torch.cuda.synchronize()

# ── Benchmark Teacher ─────────────────────────────────────────────────────────
torch.cuda.synchronize()
t_start = time.time()
for _ in range(N_BENCH):
    z = torch.randn(1, 3, H, W, device=device)
    ddim_sample_x0(teacher, z, step_size=1)
    torch.cuda.synchronize()
t_teacher = (time.time() - t_start) / N_BENCH

# ── Benchmark Student ─────────────────────────────────────────────────────────
torch.cuda.synchronize()
t_start = time.time()
for _ in range(N_BENCH):
    z = torch.randn(1, 3, H, W, device=device)
    with torch.no_grad():
        G(z)
    torch.cuda.synchronize()
t_student = (time.time() - t_start) / N_BENCH

speedup = t_teacher / t_student

print(f'Temps moyen / image (sur {N_BENCH} essais) :')
print(f'  Teacher (50 étapes DDIM) : {t_teacher*1000:.1f} ms')
print(f'  Student ( 1 étape GAN)   : {t_student*1000:.1f} ms')
print(f'  ─────────────────────────────────')
print(f'  Speedup                  : ×{speedup:.1f}')

## 6. Comparaison visuelle

On génère **N_COMPARE images** avec les **mêmes bruits z_T** pour les deux modèles.  
La grille est organisée en 3 lignes :
1. **z_T** — bruit d'entrée (même pour les deux)
2. **Teacher** — sortie 50 étapes DDIM (référence)
3. **Student** — sortie 1 étape GAN (distillé)

In [ ]:
N_COMPARE = 6

# Générer N_COMPARE z_T fixes pour la comparaison
torch.manual_seed(42)
fixed_z = torch.randn(N_COMPARE, 3, H, W, device=device)

# Générations teacher
x_teacher_list = []
for i in range(N_COMPARE):
    x_t = ddim_sample_x0(teacher, fixed_z[i:i+1], step_size=1)
    x_teacher_list.append(x_t[0].cpu())

# Générations student
with torch.no_grad():
    x_student = G(fixed_z).cpu()


def to_uint8(t):
    """Convertit un tensor [-1,1] en numpy uint8 (H,W,3)."""
    return ((t.clamp(-1, 1) + 1) / 2 * 255).byte().permute(1, 2, 0).numpy()


fig, axes = plt.subplots(3, N_COMPARE, figsize=(3.2 * N_COMPARE, 11))
fig.suptitle(
    f'Teacher (50 étapes DDIM)  vs  Student (1 étape GAN)\n'
    f'Mêmes z_T  |  Speedup ×{speedup:.1f}',
    fontsize=13, fontweight='bold'
)

row_labels = ['z_T  (bruit commun)', 'Teacher  (50 étapes)', 'Student  (1 étape)']
row_colors = ['gray', 'steelblue', 'darkorange']

for col in range(N_COMPARE):
    # Ligne 0 : bruit z_T
    z_vis = fixed_z[col].cpu().permute(1, 2, 0).numpy()
    z_vis = (z_vis - z_vis.min()) / (z_vis.max() - z_vis.min())
    axes[0, col].imshow(z_vis)
    axes[0, col].axis('off')

    # Ligne 1 : teacher
    axes[1, col].imshow(to_uint8(x_teacher_list[col]))
    axes[1, col].axis('off')

    # Ligne 2 : student
    axes[2, col].imshow(to_uint8(x_student[col]))
    axes[2, col].axis('off')

for row, (label, color) in enumerate(zip(row_labels, row_colors)):
    axes[row, 0].set_ylabel(label, fontsize=10, color=color, fontweight='bold', labelpad=8)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'teacher_vs_student.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {OUT_DIR}/teacher_vs_student.png')

## 7. Métriques quantitatives

On mesure la distance entre les sorties teacher et student sur **N_METRICS images** :

| Métrique | Description | Valeur idéale |
|----------|-------------|---------------|
| **L1** | Distance pixel moyenne | → 0 |
| **LPIPS** | Distance perceptuelle (VGG) | → 0 |

> Ces métriques mesurent l'écart entre teacher et student — **pas la qualité absolue**.  
> Un LPIPS faible signifie que le student imite bien le teacher.

In [ ]:
N_METRICS = 50

lpips_fn = lpips.LPIPS(net='vgg').to(device)
for p in lpips_fn.parameters():
    p.requires_grad = False

l1_scores    = []
lpips_scores = []

torch.manual_seed(0)
for i in range(N_METRICS):
    z = torch.randn(1, 3, H, W, device=device)

    x_t = ddim_sample_x0(teacher, z, step_size=1)   # teacher output
    with torch.no_grad():
        x_s = G(z)                                    # student output

    l1_scores.append(F.l1_loss(x_s, x_t).item())
    with torch.no_grad():
        lpips_scores.append(lpips_fn(x_s, x_t).item())

    if (i + 1) % 10 == 0:
        print(f'{i+1}/{N_METRICS}...')

print(f'\nMétriques student vs teacher (sur {N_METRICS} paires) :')
print(f'  L1 moyen    : {np.mean(l1_scores):.4f}  ± {np.std(l1_scores):.4f}')
print(f'  LPIPS moyen : {np.mean(lpips_scores):.4f}  ± {np.std(lpips_scores):.4f}')

## 8. Résumé du projet

Récapitulatif final : what we built, numbers, and tradeoffs.

In [ ]:
print('=' * 60)
print('  SINGLE-SOURCE FAST GENERATION — Résumé')
print('=' * 60)
print(f'  Image source          : {IMAGE_NAME}  ({W}×{H} px)')
print(f'')
print(f'  Teacher (SinFusion DDPM)')
print(f'    Paramètres          : {sum(p.numel() for p in teacher.parameters())/1e6:.2f} M')
print(f'    Étapes inférence    : 50 (DDIM)')
print(f'    Temps / image       : {t_teacher*1000:.1f} ms')
print(f'')
print(f'  Student (GAN distillé, Diffusion2GAN)')
print(f'    Paramètres          : {sum(p.numel() for p in G.parameters())/1e6:.2f} M')
print(f'    Étapes inférence    : 1')
print(f'    Temps / image       : {t_student*1000:.1f} ms')
print(f'')
print(f'  Speedup               : ×{speedup:.1f}')
print(f'  L1 (student vs teacher) : {np.mean(l1_scores):.4f}')
print(f'  LPIPS (student vs teacher) : {np.mean(lpips_scores):.4f}')
print('=' * 60)